In [ ]:
##this is the analysis cell for Dec dual looming assay
##batch analysis across different stimulus types
##output ploting decerleration

## any question: check on chiyu's push commit 9f74181 on the github, which was deleted for some technical reasons
import os, fnmatch
import pandas as pd
import numpy as np
import re
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
#def generate_preliminary_graph(myList = [], *args):
csv_pattern='trial_*.csv'
database_pattern='*.pickle'
exp_name='sweeploom'
frame_rate=144
#stim_duration=3
grey_duration=1
#analysis_window=frame_rate*(stim_duration+grey_duration)
analysis_window=[-2, 5]
window_length=analysis_window[1]-analysis_window[0]
stim_type=["c2b_f_sloom","c2b_s_sloom","c2f_f_sloom","c2f_s_sloom","c2c_f_loom","c2c_s_loom"]
save_fig = True
closed_loop_experiment = False
sg_filering= True
analyse_every_trial = False
plotting_ThisExperiment_trajectory = False
#plotting_ThisTrial_trajectory = True ##this is an option to plot individual trajectory of each trial
##loop through the entire list of animals
fig, (ax1,ax2) = plt.subplots(nrows=1, ncols=2, figsize=(18,7),tight_layout=True)
for thisDir in myList:
    mean_travel_distance_during_stim=[]
    mean_normalised_distance_during_stim=[]
    trial_num=[]
    travel_distance_prior_stim_all=[]
    travel_distance_tbt=[]
    velocity_list=[]
    distance_change_t1=[]
    distance_change_t2=[]
    stimulus_order=[]
    deceleration_across_stim=[]
    files = os.listdir(thisDir)
    db_file= fnmatch.filter(files, database_pattern)
    df=pd.read_pickle(os.path.join(thisDir,db_file[0]))
    if sg_filering == True:
        x_all = savgol_filter(df.loc[:, ['intergrated x position']], 71, 3, axis=0)
        y_all = savgol_filter(df.loc[:, ['intergrated y position']], 71, 3, axis=0)
        instant_v=savgol_filter(df.loc[:, ['instant speed']], 39, 3, axis=0)
        runmean_v=df['instant speed'].rolling(10).mean().values
        #vx=savgol_filter(df.loc[:, ['instant speed']], 39, 3, axis=0)
        #vz=savgol_filter(df.loc[:, ['instant speed']], 39, 3, axis=0)
    else:
        x_all = df.loc[:, ['intergrated x position']]
        y_all = df.loc[:, ['intergrated y position']]
        instant_v=df.loc[:, ['instant speed']]

    if plotting_ThisExperiment_trajectory == True:
        plt.plot(x_all, y_all,c=np.arange(len(y_all)), marker='.', alpha=0.5)
        plt.show()   

    csv_file= fnmatch.filter(files, csv_pattern)
    stim_info = pd.read_csv(os.path.join(thisDir ,csv_file[0]))
    stim_info=stim_info.reset_index()
    stim_variable=[]
    stimulus_timestamp=[]

    if exp_name == 'gratings':
        for row in range(0, len(stim_info)):
        #delete the bracket in current speed and save it as integer in new list
            stim_variable.append(int(stim_info.loc[row, 'Frame'].replace('(', '')))
        # combine the last two values into the timestamp (were divided into 2 columns cause german computer
        # used comma as decimal seperator and save timestamp as float in new file
            if isinstance(stim_info.loc[row, 'Value'], float):
                timestamp = stim_info.loc[row, 'Timestamp']
            else:
                timestamp = str(stim_info.loc[row, 'Timestamp']) + '.' + stim_info.loc[row, 'Value']
            stimulus_timestamp.append(float(timestamp.replace(')', '')))
        stim_variable=np.array(stim_variable).reshape((len(stim_info), -1))

    elif exp_name == 'sweeploom':
        for row in range(0, len(stim_info)):
            stim_variable.append(int(stim_info.iloc[row,2].split("=",1)[1]))
            stim_variable.append(int(stim_info.iloc[row,3].split("=",1)[1]))
            stim_variable.append(int(stim_info.iloc[row,4].split("=",1)[1]))
            stim_variable.append(int(stim_info.iloc[row,5].split("=",1)[1]))
            stim_variable.append(int(stim_info.iloc[row,6].split("=",1)[1]))
            stim_variable.append(int(stim_info.iloc[row,7].split("=",1)[1]))
            stim_variable.append(int(stim_info.iloc[row,8].split("=",1)[1]))
            #stim_variable.append(int(re.split(r'=|}',stim_info.iloc[row,5])[1]))
            if isinstance(stim_info.loc[row, 'Value'], float):
                timestamp = stim_info.loc[row, 'Timestamp']
            else:
                timestamp = str(stim_info.loc[row, 'Timestamp']) + '.' + stim_info.loc[row, 'Value']
            stimulus_timestamp.append(float(timestamp.replace(')', '')))
        stim_variable=np.array(stim_variable).reshape((len(stim_info), -1))
        #some bugs in the bonsai workflow. Needs to remove the first timestamp and the last row of stim variable here
        if closed_loop_experiment == True:
            stimulus_timestamp.pop(0)
            stim_variable=np.delete(stim_variable, -1, axis=0)
    elif exp_name == 'dots':
        for row in range(0, len(stim_info)):
            stim_variable.append(int(stim_info.iloc[row,2].split("=",1)[1]))
            stim_variable.append(int(stim_info.iloc[row,4].split("=",1)[1]))
            stim_variable.append(int(stim_info.iloc[row,5].split("=",1)[1]))
            stim_variable.append(int(stim_info.iloc[row,7].split("=",1)[1]))
            stim_variable.append(int(stim_info.iloc[row,11].split("=",1)[1]))
            stim_variable.append(int(stim_info.iloc[row,13].split("=",1)[1]))
            stim_variable.append(int(stim_info.iloc[row,17].split("=",1)[1]))
            if isinstance(stim_info.loc[row, 'Value'], float):
                timestamp = stim_info.loc[row, 'Timestamp']
            else:
                timestamp = str(stim_info.loc[row, 'Timestamp']) + '.' + stim_info.loc[row, 'Value']
            stimulus_timestamp.append(float(timestamp.replace(')', '')))
        stim_variable=np.array(stim_variable).reshape((len(stim_info), -1))
    else:
        warnings.warn('stimulus paradigm can not be recognised')

    stim_df=pd.DataFrame(stim_variable)    

        # compare the timestamp between event file and dat file
    data_timestamp=df.loc[:, 'timestamp']
    stimulus_timestamp_b=np.tile(stimulus_timestamp,(len(data_timestamp),1))
    data_timestamp_b=np.tile(data_timestamp,(len(stimulus_timestamp),1))
    timestamp_diff= data_timestamp_b > np.transpose(stimulus_timestamp_b)
    stim_onset= np.where(np.roll(timestamp_diff,1,axis=1)!=timestamp_diff)[1][1::2]
    stim_df["stim_onset"]=stim_onset

    filters=[(stim_df.iloc[:,6]==1) & (stim_df.iloc[:,2] > stim_df.iloc[:,3]),
    (stim_df.iloc[:,6]==2) & (stim_df.iloc[:,2] > stim_df.iloc[:,3]),
    (stim_df.iloc[:,6]==1) & (stim_df.iloc[:,2] < stim_df.iloc[:,3]),
    (stim_df.iloc[:,6]==2) & (stim_df.iloc[:,2] < stim_df.iloc[:,3]),
    (stim_df.iloc[:,6]==1) & (stim_df.iloc[:,2] == stim_df.iloc[:,3]),
    (stim_df.iloc[:,6]==2) & (stim_df.iloc[:,2] == stim_df.iloc[:,3])]

    stim_df["stim_type"]=np.select(filters, stim_type)
    for thisStim in stim_type:
        if analyse_every_trial==True:
            stim_to_analyse=stim_df.loc[:,'stim_onset']
            stim_name_list=np.array(stim_df["stim_type"])
        #elif: 'myVar' in locals():
        #stim_type=["m2b_f_sloom","m2b_s_sloom","m2f_f_sloom","m2f_s_sloom","m2m_f_loom","m2m_s_loom"]
        else:
            stim_to_analyse=stim_df.loc[np.where((stim_df.loc[:,'stim_type']== thisStim)),"stim_onset"]
            stim_name_list=stim_df.loc[stim_to_analyse.index,'stim_type'].values
        if len(stim_to_analyse) ==0:
            print('no_required_stim_type')
            #continue

        if re.search('_f_', thisStim):##create this condition so that stim_duration became dynamics
            stim_duration=3
            presence_duration=1
            sweep_duration=1
        elif re.search('_s_', thisStim):
            stim_duration=4
            presence_duration=1
            sweep_duration=2
        else:
            print('something wrong with the stim list')


        grey_onset=stim_to_analyse+np.ones(len(stim_to_analyse),dtype=int)*analysis_window[0]*frame_rate
        analysie_window_tile=np.tile(np.arange(window_length*frame_rate),(len(grey_onset),1))
        index_points=np.array(grey_onset).reshape(len(grey_onset),1)+analysie_window_tile
        x=x_all[index_points]
        y=y_all[index_points]
        x_dif=np.diff(x,axis=1)
        y_dif=np.diff(y,axis=1)
        travel_distance_fbf=np.sqrt(np.add(np.square(x_dif), np.square(y_dif)))
        velocity=travel_distance_fbf*frame_rate
        cum_travel_distance=np.cumsum(np.sqrt(np.add(np.square(x_dif), np.square(y_dif))),axis=1)
        init_time=abs(analysis_window[0])
        end_time=abs(analysis_window[0])+stim_duration
        travel_distance_prior_stim=cum_travel_distance[:,init_time*frame_rate,0]
        travel_distance_post_stim=cum_travel_distance[:,end_time*frame_rate,0]

        travel_distance_after_stim_presence=cum_travel_distance[:,(init_time+presence_duration)*frame_rate,0]
        travel_distance_after_stim_sweep=cum_travel_distance[:,(init_time+presence_duration+sweep_duration)*frame_rate,0]
        travel_distance_during_stim_presence=travel_distance_after_stim_presence-travel_distance_prior_stim
        travel_distance_during_stim_sweep=travel_distance_after_stim_sweep-travel_distance_after_stim_presence
        normalised_distance_change=np.divide(travel_distance_during_stim_sweep,travel_distance_during_stim_presence)

        mean_travel_distance_post_stim=np.mean(travel_distance_post_stim)
        travel_distance_during_stim=travel_distance_post_stim-travel_distance_prior_stim
        normalise_travel_distance=np.divide(travel_distance_post_stim,travel_distance_prior_stim)  
        mean_normalise_travel_distance=np.mean(normalise_travel_distance)

        travel_distance_1sec_prior_stim=cum_travel_distance[:,init_time*frame_rate,0]-cum_travel_distance[:,(init_time-1)*frame_rate,0]
        running_trial=travel_distance_1sec_prior_stim>5
        running_velocity=velocity[running_trial,:]
        runnning_trial_distance_change=normalised_distance_change[running_trial]
        travel_distance_1sec_post_stim=cum_travel_distance[running_trial,(init_time+1)*frame_rate,0]-cum_travel_distance[running_trial,init_time*frame_rate,0]
        velocity_type1=running_velocity[travel_distance_1sec_post_stim>5,:]
        velocity_type2=running_velocity[travel_distance_1sec_post_stim<5,:]
        runnning_trial_distance_change_type1=runnning_trial_distance_change[travel_distance_1sec_post_stim>5]
        runnning_trial_distance_change_type2=runnning_trial_distance_change[travel_distance_1sec_post_stim<5] 
        
        #use deceleration_detection function which returns the index of onset but in a reverse order
        detected_epochs=deceleration_detection(velocity_type1)
        deceleration_frame_num_in_the_VOI=frame_rate-detected_epochs[1]
        deceleration_onset=0.5*presence_duration+deceleration_frame_num_in_the_VOI/frame_rate
        
        ##started to compile everything across stimulus type
        deceleration_across_stim.append(deceleration_onset)
        stimulus_order.append(sweep_duration)
        travel_distance_prior_stim_all.append(travel_distance_prior_stim)
        travel_distance_tbt.append(cum_travel_distance)
        distance_change_t1.append(runnning_trial_distance_change_type1)
        distance_change_t2.append(runnning_trial_distance_change_type1)
        ## please tell me why I wanted to take mean here
        mean_travel_distance_during_stim.append(np.mean(travel_distance_during_stim))
        mean_normalised_distance_during_stim.append(np.mean(normalise_travel_distance))
    ##concatenate travel distance for histogram
    travel_distance_tbt_prior_stim=np.concatenate(travel_distance_prior_stim_all)
    travel_distance_tbt_during_analysis_window=np.concatenate(travel_distance_tbt)
    distance_change_tbt_acorss_experiment=np.concatenate(distance_change_t1)
    ##concatenate velocity list for heatmap
    #velocity_tbt=np.vstack(velocity_list)
    ##arrange xticks at onset and offset of the stimulus for heatmap
    #heatmap_xticks=[abs(analysis_window[0])*frame_rate, (stim_duration+abs(analysis_window[0]))*frame_rate]
    #trial_num_across_stim=np.cumsum(trial_num)

    dates=[1,2,3]
    values=[np.mean(deceleration_across_stim[1]),np.mean(deceleration_across_stim[5]),np.mean(deceleration_across_stim[3])]
    yerr=[np.std(deceleration_across_stim[1], axis=0, ddof=1)/np.sqrt(deceleration_across_stim[1].shape[0]),np.std(deceleration_across_stim[5], axis=0, ddof=1) / np.sqrt(deceleration_across_stim[5].shape[0]),np.std(deceleration_across_stim[3], axis=0, ddof=1) / np.sqrt(deceleration_across_stim[3].shape[0])]
    ax1.plot(dates, values, 'o-') 
    ax1.errorbar(dates, values,yerr=yerr,fmt='-o')
    values=[np.mean(deceleration_across_stim[0]),np.mean(deceleration_across_stim[4]),np.mean(deceleration_across_stim[2])]
    yerr=[np.std(deceleration_across_stim[0], axis=0, ddof=1)/np.sqrt(deceleration_across_stim[0].shape[0]),np.std(deceleration_across_stim[4], axis=0, ddof=1) / np.sqrt(deceleration_across_stim[4].shape[0]),np.std(deceleration_across_stim[2], axis=0, ddof=1) / np.sqrt(deceleration_across_stim[2].shape[0])]
    ax2.plot(dates, values, 'o-')
    ax2.errorbar(dates, values,yerr=yerr,fmt='-o')


    ax1.set_xlabel('stimulus type')
    ax1.set_ylabel('Deceleration onset (s)')
    ax2.set_xlabel('stimulus type')
    ax2.set_ylabel('Deceleration onset (s)')
    ax1.set_title('slow stimulus')
    ax2.set_title('fast stimulus')
    ax1.set_ylim(0,1.5)
    ax2.set_ylim(0,1.5)
    ax1.set_xticks([1,2,3])
    ax1.set_xticklabels(['back','center','front'])
    ax2.set_xticks([1,2,3])
    ax2.set_xticklabels(['back','center','front'])
    ax1.set_xlim(0,4)
    ax2.set_xlim(0,4)

plot_file_name=f'decer_during_05st_sec_sweep.png'
if save_fig == True:
    plt.savefig(os.path.join('C:/Users/neuroPC/Desktop/data',plot_file_name))

In [ ]:
 ## additional plotting function
## to plot a histogram of speed distribution and heatmap of speed
    
    fig, (ax1,ax2) = plt.subplots(nrows=1, ncols=2, figsize=(18,7),tight_layout=True)
        ax1.hist(travel_distance_tbt_prior_stim, bins=20)
        ax1.set_xlabel('Speed (mm/s)')
        ax1.set_ylabel('Times')
        ax1.set_title('speed distribution before stim onset')
        ## draw subplot2
        #pcm=ax2.pcolormesh(velocity_tbt, cmap = 'magma',vmin = 0, vmax =100)
        pcm=ax2.pcolormesh(velocity_tbt, cmap = 'magma') ##sometimes, we dont need a vmax
        ## add colorbar
        fig.colorbar(pcm)
        ## add horizontal lines to indicate each stimulus type
        for ind in range(0,trial_num_across_stim.shape[0]):
            ax2.axhline(y=trial_num_across_stim[ind], color='white', linestyle='-', label='Stimulus Type',linewidth=0.5)
        ## remove y ticks for clearity and add y label
        ax2.set_yticks([])
        ax2.set_ylabel('Trial#')
        ## add vertical lines to indicate stimulus onset and offset
        ax2.axvline(x=heatmap_xticks[0], color='white', linestyle='-', label='Stimulus Starts')
        ax2.axvline(x=heatmap_xticks[1], color='white', linestyle='--', label='Fast Stimulus Ends')
        ax2.axvline(x=heatmap_xticks[1]+1*frame_rate, color='white', linestyle='--', label='Slow Stimulus Ends')
        ax2.set_xticks([heatmap_xticks[0],heatmap_xticks[1]])
        ax2.set_xticklabels(['0','3'])
        ax2.set_xlabel('Time (s)')
        ## add title
        ax2.set_title('stim_related response tbt')
        ## save the file

        plot_file_name=f'stim_heatmap_from_{str(analysis_window[0])}_to_{str(analysis_window[1])}s.png'
        if save_fig == True:
            plt.savefig(os.path.join(thisDir,plot_file_name))
        
    return velocity_tbt, travel_distance_tbt_during_analysis_window

In [ ]:
## just a cell to store different criteria

filters=[(stim_df.iloc[:,6]==1) & (stim_df.iloc[:,2] > stim_df.iloc[:,3]),
(stim_df.iloc[:,6]==2) & (stim_df.iloc[:,2] > stim_df.iloc[:,3]),
(stim_df.iloc[:,6]==1) & (stim_df.iloc[:,2] < stim_df.iloc[:,3]),
(stim_df.iloc[:,6]==2) & (stim_df.iloc[:,2] < stim_df.iloc[:,3]),
(stim_df.iloc[:,6]==1) & (stim_df.iloc[:,2] == stim_df.iloc[:,3]),
(stim_df.iloc[:,6]==2) & (stim_df.iloc[:,2] == stim_df.iloc[:,3])]


### use this criteria for experiments during exp_date=['221015','221016']
#filters=[(stim_df.iloc[:,0]==1) & (stim_df.iloc[:,2] > stim_df.iloc[:,3]),
#(stim_df.iloc[:,0]==8) & (stim_df.iloc[:,2] > stim_df.iloc[:,3]),
#(stim_df.iloc[:,0]==1) & (stim_df.iloc[:,2] < stim_df.iloc[:,3]),
#(stim_df.iloc[:,0]==8) & (stim_df.iloc[:,2] < stim_df.iloc[:,3]),
#(stim_df.iloc[:,0]==1) & (stim_df.iloc[:,2] == stim_df.iloc[:,3]),
#(stim_df.iloc[:,0]==8) & (stim_df.iloc[:,2] == stim_df.iloc[:,3])]

### add new column here to store event info
#         raw_data['frame_rate']=frame_rate
#         if isinstance(frame_rate, int):
#             raw_data['frame_rate']=raw_data['frame_rate'].astype('int16')  
#         elif isinstance(frame_rate, int):
#             raw_data['frame_rate']=raw_data['frame_rate'].astype('float16')

stim_df["stim_type"]=np.select(filters, stim_type)